In [6]:
import pdfplumber
import pandas as pd

rows = []

with pdfplumber.open("Constituents_ENG_02014.pdf") as pdf:

    for page in pdf.pages:

        table = page.extract_table()

        if table is None:
            continue

        # Skip the header row
        rows.extend(table[1:])

columns = [
    "Date",
    "Index",
    "Stock Code",
    "Stock Name",
    "ISIN",
    "Exchange",
    "Industry",
    "Currency",
    "Closing Price",
    "Daily Change"
]

constituents = pd.DataFrame(rows, columns=columns)

print(constituents.head())
print(f"\nRows: {len(constituents)}")

         Date                                         Index Stock Code  \
0  27-07-2026  Hang Seng Composite LargeCap &\nMidCap Index        883   
1  27-07-2026  Hang Seng Composite LargeCap &\nMidCap Index        857   
2  27-07-2026  Hang Seng Composite LargeCap &\nMidCap Index       1088   
3  27-07-2026  Hang Seng Composite LargeCap &\nMidCap Index        386   
4  27-07-2026  Hang Seng Composite LargeCap &\nMidCap Index       1171   

           Stock Name          ISIN                   Exchange Industry  \
0               CNOOC  HK0883013259  Hong Kong\nStock Exchange   Energy   
1          PETROCHINA  CNE1000003W8  Hong Kong\nStock Exchange   Energy   
2       CHINA SHENHUA  CNE1000002R0  Hong Kong\nStock Exchange   Energy   
3        SINOPEC CORP  CNE1000002Q2  Hong Kong\nStock Exchange   Energy   
4  YANKUANG ENER-\nGY  CNE1000004Q8  Hong Kong\nStock Exchange   Energy   

  Currency Closing Price Daily Change  
0      HKD         22.98        -0.95  
1      HKD          9.84

In [7]:
constituents["Ticker"] = (
    constituents["Stock Code"]
    .astype(int)
    .astype(str)
    .str.zfill(4)
    + ".HK"
)

print(constituents[["Stock Code","Ticker"]].head())

  Stock Code   Ticker
0        883  0883.HK
1        857  0857.HK
2       1088  1088.HK
3        386  0386.HK
4       1171  1171.HK


In [9]:
#verify
tickers = constituents["Ticker"].unique().tolist()

print(len(tickers))
print(tickers[:20])

326
['0883.HK', '0857.HK', '1088.HK', '0386.HK', '1171.HK', '1898.HK', '3668.HK', '2883.HK', '3899.HK', '0467.HK', '9988.HK', '3690.HK', '1211.HK', '9618.HK', '0669.HK', '9888.HK', '1024.HK', '9987.HK', '0175.HK', '9992.HK']


In [10]:
import yfinance as yf
import pandas as pd

all_prices = []

for ticker in tickers:

    print(f"Downloading {ticker}")

    try:

        df = yf.download(
            ticker,
            period="5y",
            interval="1d",
            auto_adjust=False,
            progress=False
        )

        if df.empty:
            print(f"No data for {ticker}")
            continue

        # Flatten MultiIndex columns if present
        if isinstance(df.columns, pd.MultiIndex):
            df.columns = df.columns.get_level_values(0)

        df = df.reset_index()

        df["Ticker"] = ticker

        all_prices.append(df)

    except Exception as e:
        print(f"{ticker}: {e}")

market = pd.concat(all_prices, ignore_index=True)

market.to_parquet("hk_market_5y.parquet", index=False)

print(market.head())

ImportError: Unable to find a usable engine; tried using: 'pyarrow', 'fastparquet'.
A suitable version of pyarrow or fastparquet is required for parquet support.
Trying to import the above resulted in these errors:
 - `Import pyarrow` failed. pyarrow is required for parquet support. Use pip or conda to install the pyarrow package.
 - `Import fastparquet` failed. fastparquet is required for parquet support. Use pip or conda to install the fastparquet package.

In [11]:
print(market.shape)
market.head()

(352201, 8)


Price,Date,Adj Close,Close,High,Low,Open,Volume,Ticker
0,2021-07-28,4.798914,7.74,7.92,7.71,7.88,114313426,0883.HK
1,2021-07-29,4.829915,7.79,7.90,7.70,7.81,156777656,0883.HK
2,2021-07-30,4.774114,7.70,7.79,7.59,7.78,115283717,0883.HK
3,2021-08-02,4.836115,7.80,7.92,7.63,7.70,92437585,0883.HK
4,2021-08-03,4.836115,7.80,7.89,7.70,7.78,74495211,0883.HK


In [13]:
print("Unique tickers:", market["Ticker"].nunique())

Unique tickers: 326


In [14]:
market.isnull().sum()

Price
Date           0
Adj Close    325
Close        325
High           0
Low            0
Open           0
Volume         0
Ticker         0
dtype: int64

In [15]:
counts = market.groupby("Ticker").size()

print(counts.describe())

print("\nSmallest histories:")
print(counts.sort_values().head(20))

count     326.000000
mean     1080.371166
std       327.611844
min       108.000000
25%      1227.000000
50%      1227.000000
75%      1227.000000
max      1227.000000
dtype: float64

Smallest histories:
Ticker
2706.HK    108
1768.HK    120
0100.HK    133
2675.HK    134
2513.HK    134
9903.HK    134
6082.HK    138
7618.HK    151
2788.HK    164
2698.HK    174
0800.HK    176
2026.HK    176
2718.HK    179
1384.HK    182
2259.HK    200
9973.HK    203
2525.HK    210
2580.HK    220
2591.HK    232
2590.HK    259
dtype: int64


In [16]:
duplicates = market.duplicated(subset=["Date", "Ticker"]).sum()

print("Duplicate rows:", duplicates)

Duplicate rows: 0


In [18]:
market[market["Ticker"] == "0700.HK"].head()

Price,Date,Adj Close,Close,High,Low,Open,Volume,Ticker
182496,2021-07-28,395.731171,412.254211,424.607086,389.023407,415.757263,122431259,0700.HK
182497,2021-07-29,435.375092,453.553375,455.581451,431.981934,442.491089,79075566,0700.HK
182498,2021-07-30,423.871277,441.569244,449.497192,428.663239,449.497192,43889700,0700.HK
182499,2021-08-02,420.331604,437.881805,441.384857,423.316467,438.803680,31941064,0700.HK
182500,2021-08-03,394.669281,411.147980,432.166290,390.498383,432.166290,127799280,0700.HK


In [19]:
market = market.rename(columns={
    "Date": "date",
    "Ticker": "ticker",
    "Open": "open",
    "High": "high",
    "Low": "low",
    "Close": "close",
    "Adj Close": "adj_close",
    "Volume": "volume"
})

In [20]:
market["market"] = "HK"
market["exchange"] = "HKEX"
market["board"] = "Main Board"

In [21]:
#previous close
market = market.sort_values(["ticker", "date"])

market["prev_close"] = (
    market
    .groupby("ticker")["close"]
    .shift(1)
)

In [22]:
#pct change
market["pct_change"] = (
    (market["close"] - market["prev_close"])
    / market["prev_close"]
    * 100
)

In [23]:
#zero volume
market["is_zero_or_missing_volume"] = (
    market["volume"].isna()
    | (market["volume"] == 0)
)

In [25]:
#aproximation of suspended
market["is_suspended"] = (
    market["volume"].isna()
    | (market["volume"] == 0)
)

In [27]:
#those columns don't exist on HK, only chinese market
market["is_st"] = False
market["limit_pct"] = None
market["limit_up_price"] = None
market["limit_down_price"] = None
market["is_limit_up"] = False
market["is_limit_down"] = False

In [28]:
market.to_csv("hk_market_5y.csv", index=False)